# SteadyState — steady state with a spatially varying diffusivity (2D + 3D)

`MagneticStandalone` · ASN 28 validation case (TRUST-LiMe).


## Introduction

Validation of the **standalone magnetic (induction) equation** added by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data (no Navier–Stokes coupling in this first phase). Every case is
**self-contained** (inline `Mailler`/`Pave` mesh, no `.med` file) and records
point probes (`Postraitement/Sondes`) into `.son` files which are compared
below against a closed-form solution.

This notebook runs the 2D and 3D versions of this test case: ``steady_state``. Per the TRUST
validation-form layout (`Validation/Rapports_automatiques/<Category>/<FormName>/`),
the other `MagneticStandalone` cases live in their own sibling folders, each
with `<FormName>/<FormName>.ipynb` + `src/<case>.data`.


Both dimensions are run from a single shared data template in this form; the 3D mesh is converted with `Polyedriser dom`.
## What it validates

$b=y$ with $\eta(x)=1+x$ and $u=0$. Since $\nabla b=e_y$,
$\nabla\cdot(\eta\nabla b)=\partial_y(1+x)=0$: the profile is an exact steady
solution for any diffusivity that varies only in $x$. Any spurious source or
mis-assembled diffusion operator destroys it, and the diffusivity is read from a
**functional** field, exercising the non-uniform-$\eta$ path.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| 6 | diffusion in divergence form $\nabla\cdot(\eta\nabla b)$ | $\eta=1+x$ equilibrium |


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name, subdir=None):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    if subdir is None:
        subdir = "3D/coarse" if "_3d" in case.lower() else "coarse"
    path = f"{BUILD}/{subdir}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def ratio_error(case, tags, ref):
    """Max |b(t)/b(0) - ref(t)| over the given probes and all recorded times."""
    worst = 0.0
    for tag in tags:
        t, b = read_son(case, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - ref(t)))))
    return worst

print("BUILD =", BUILD)


## Run both dimensions

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).


In [ ]:
from string import Template
from trustutils import run

NB_PROCS = 2
run.reset()

# Mesh geometries and physics/post-processing blocks are the two dimensions
# of the same src template. The 3D case uses a distinct target data name
# and build subtree, so partition zones and probe files cannot collide.
MESH_BLOCKS = {
    2: r"""Mailler dom
{
    Pave pavé
    {
        Origine              0.  0.
        Nombre_de_Noeuds     $nx $ny
        Longueurs            1.  1.
    }
    {
        Bord gauche   X = 0.    0. <= Y <= 1.
        Bord droite   X = 1.    0. <= Y <= 1.
        Bord bas      Y = 0.    0. <= X <= 1.
        Bord haut     Y = 1.    0. <= X <= 1.
    }
}""",
    3: r"""Mailler dom
{
    Pave cube
    {
        /* same y-resolution as the 2D case (33 nodes -> 32 cells), coarse x/z */
        Origine              0.  0.  0.
        Nombre_de_Noeuds     $nx $ny $nz
        Longueurs            1.  1.  1.
    }
    {
        Bord gauche   X = 0.    0. <= Y <= 1.  0. <= Z <= 1.
        Bord droite   X = 1.    0. <= Y <= 1.  0. <= Z <= 1.
        Bord bas      Y = 0.    0. <= X <= 1.  0. <= Z <= 1.
        Bord haut     Y = 1.    0. <= X <= 1.  0. <= Z <= 1.
        Bord devant   Z = 0.    0. <= X <= 1.  0. <= Y <= 1.
        Bord derriere Z = 1.    0. <= X <= 1.  0. <= Y <= 1.
    }
}

Polyedriser dom""",
}
DIMENSION_BLOCKS = {
    2: {
        'boundary_block': r"""boundary_conditions
        {
            # b = y prescribed on the two faces normal to Y #
            bas     Temperature_imposee_paroi Champ_front_uniforme 1 0.
            haut    Temperature_imposee_paroi Champ_front_uniforme 1 1.
            # zero flux on the faces normal to X (b does not depend on x) #
            gauche  Neumann_Paroi_adiabatique
            droite  Neumann_Paroi_adiabatique
        }""",
        'post_block': r"""Postraitement
  {
    Sondes
    {
      # b = y -> probes read 0.25 / 0.5 / 0.75 at all times #
      sonde_y025 grav magnetic_field Periode 1e-6 Points 1 0.5 0.25
      sonde_y050 grav magnetic_field Periode 1e-6 Points 1 0.5 0.50
      sonde_y075 grav magnetic_field Periode 1e-6 Points 1 0.5 0.75
    }
  }""",
    },
    3: {
        'boundary_block': r"""boundary_conditions
        {
            # b = y prescribed on the two faces normal to Y #
            bas     Temperature_imposee_paroi Champ_front_uniforme 1 0.
            haut    Temperature_imposee_paroi Champ_front_uniforme 1 1.
            # zero flux on the faces normal to X and Z (b does not depend on x or z) #
            gauche   Neumann_Paroi_adiabatique
            droite   Neumann_Paroi_adiabatique
            devant   Neumann_Paroi_adiabatique
            derriere Neumann_Paroi_adiabatique
        }""",
        'post_block': r"""Postraitement
  {
    Sondes
    {
      # b = y -> probes read 0.25 / 0.5 / 0.75 at all times #
      sonde_y025 grav magnetic_field Periode 1e-6 Points 1 0.5 0.25 0.5
      sonde_y050 grav magnetic_field Periode 1e-6 Points 1 0.5 0.50 0.5
      sonde_y075 grav magnetic_field Periode 1e-6 Points 1 0.5 0.75 0.5
    }
  }""",
    },
}

MESHES_2D = {'coarse': {'nx': 33, 'ny': 33}, 'standard': {'nx': 65, 'ny': 65}, 'fine': {'nx': 129, 'ny': 129}}
MESHES_3D = {'coarse': {'nx': 9, 'ny': 33, 'nz': 9}, 'standard': {'nx': 9, 'ny': 65, 'nz': 9}, 'fine': {'nx': 9, 'ny': 129, 'nz': 9}}

def register_dimension(dim, meshes, prefix="", target_data=None, nb_procs=NB_PROCS, partition=True):
    registered = {}
    for level, mesh in meshes.items():
        dic = {"dim": dim,
               "mesh_block": Template(MESH_BLOCKS[dim]).substitute(**mesh),
               **DIMENSION_BLOCKS[dim]}
        target = f"{prefix}/{level}" if prefix else level
        kwargs = {"nbProcs": nb_procs, "excluNR": True}
        if target_data is not None:
            kwargs["targetData"] = target_data
        registered[level] = run.addCaseFromTemplate('steady_state.data', target, dic=dic, **kwargs)
        if partition:
            registered[level].partition()
    return registered

cases = register_dimension(2, MESHES_2D)
cases_3d = register_dimension(3, MESHES_3D, prefix="3D", target_data='steady_state_3d.data', nb_procs=NB_PROCS, partition=True)
case = cases["coarse"]
case_3d = cases_3d["coarse"]
run.printCases()
run.runCases()


## 2D checks


In [ ]:
dy_half = 0.5 / 32.0   # 33x33 nodes -> 32 cells over [0,1]; P0 is cell-centered
for tag, y_ref in [("SONDE_Y025", 0.25), ("SONDE_Y050", 0.50), ("SONDE_Y075", 0.75)]:
    t, b = read_son(case.dataFileName_, tag)
    # the equilibrium must not evolve in time
    drift = np.max(np.abs(b - b[0]))
    # the P0 (cell-centered) projection of the linear profile b=y stands within
    # half a cell of the analytic value at every probe
    repr_err = np.abs(b[0] - y_ref)
    print(f"{tag}: b(0)={b[0]:.5f} (analytic {y_ref}), drift={drift:.2e}, P0 repr err={repr_err:.5f}")
    assert drift < 1e-8, f"steady state not preserved at {tag}: drift={drift:.3e}"
    assert repr_err <= dy_half + 1e-6, f"P0 projection off at {tag}: {repr_err:.3e}"
print("OK: equilibrium with non-uniform eta is preserved (P0 projection of b=y).")

import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
t, b = read_son(case.dataFileName_, "SONDE_Y050")
plt.plot(t, b, label="b(x=0.5, y=0.5, t)  [b=y equilibrium]")
plt.axhline(0.5, color="k", ls="--", label="analytic 0.5")
plt.xlabel("t [s]"); plt.ylabel("b"); plt.legend(); plt.grid(True); plt.show()


## 2D mesh convergence

The shared refinement ladder: `coarse` = the baseline mesh above (`33×33`),
`standard` = cells ×2 (`65×65`), `fine` = cells ×4 (`129×129`).

The equilibrium $b = y$ with non-uniform $\eta$ is a steady identity: the
drift bound $\max_t |b(t) - b(0)| < 10^{-8}$ and the P0 projection bound
$\ |b(0) - y_{\mathrm{ref}}| \le \frac{1}{2}\Delta y(L)$ (the half-cell of
the level) are asserted **at every level** — the projection bound tightens
by construction as the mesh refines, so no convergence order is reported.


In [ ]:
SONDE_TAGS_SCAN = [("SONDE_Y025", 0.25), ("SONDE_Y050", 0.50), ("SONDE_Y075", 0.75)]

def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

scan = {"coarse": ("coarse", case.dataFileName_),
        "standard": (f"standard", cases["standard"].dataFileName_),
        "fine": (f"fine", cases["fine"].dataFileName_)}
scan_nodes = {"coarse": (33, 33), "standard": (65, 65), "fine": (129, 129)}

print(f"level  nodes      max drift      max P0 repr err (bound)")
for lvl in ("coarse", "standard", "fine"):
    nx, ny = scan_nodes[lvl]
    dy_half = 0.5 / (ny - 1)          # half-cell of this level
    worst_drift, worst_repr = 0.0, 0.0
    for tag, y_ref in SONDE_TAGS_SCAN:
        _, b = read_son_dir(scan[lvl][0], scan[lvl][1], tag)
        worst_drift = max(worst_drift, float(np.max(np.abs(b - b[0]))))
        worst_repr = max(worst_repr, abs(float(b[0]) - y_ref))
    print(f"{lvl:<9} {nx:>4}x{ny:<4} {worst_drift:<13.2e} {worst_repr:.5f} ({dy_half + 1e-6:.5f})")
    assert worst_drift < 1e-8, f"equilibrium drift at {lvl}: {worst_drift:.3e}"
    assert worst_repr <= dy_half + 1e-6, f"P0 projection off at {lvl}: {worst_repr:.3e}"
print("OK: steady identity and the tightening P0 bound hold at every level.")

## 3D checks


In [ ]:
dy_half = 0.5 / 32.0   # 33x33 nodes -> 32 cells over [0,1]; P0 is cell-centered
for tag, y_ref in [("SONDE_Y025", 0.25), ("SONDE_Y050", 0.50), ("SONDE_Y075", 0.75)]:
    t, b = read_son(case_3d.dataFileName_, tag)
    # the equilibrium must not evolve in time
    drift = np.max(np.abs(b - b[0]))
    # the P0 (cell-centered) projection of the linear profile b=y stands within
    # half a cell of the analytic value at every probe
    repr_err = np.abs(b[0] - y_ref)
    print(f"{tag}: b(0)={b[0]:.5f} (analytic {y_ref}), drift={drift:.2e}, P0 repr err={repr_err:.5f}")
    assert drift < 1e-8, f"steady state not preserved at {tag}: drift={drift:.3e}"
    assert repr_err <= dy_half + 1e-6, f"P0 projection off at {tag}: {repr_err:.3e}"
print("OK: equilibrium with non-uniform eta is preserved (P0 projection of b=y).")

import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
t, b = read_son(case_3d.dataFileName_, "SONDE_Y050")
plt.plot(t, b, label="b(x=0.5, y=0.5, t)  [b=y equilibrium]")
plt.axhline(0.5, color="k", ls="--", label="analytic 0.5")
plt.xlabel("t [s]"); plt.ylabel("b"); plt.legend(); plt.grid(True); plt.show()


## 3D mesh convergence

The shared refinement ladder, adapted to the 1D equilibrium physics: only
the active `y` direction is refined (`coarse` = `9×33×9`, `standard` = y cells ×2
(`9×65×9`), `fine` = y cells ×4 (`9×129×9`)).

The equilibrium $b = y$ with non-uniform $\eta$ is a steady identity: the
drift bound $\max_t |b(t) - b(0)| < 10^{-8}$ and the P0 projection bound
$|b(0) - y_{\mathrm{ref}}| \le \frac{1}{2}\Delta y(L)$ (the half-cell of
the level) are asserted **at every level** — the projection bound tightens
by construction as the mesh refines, so no convergence order is reported.

In [ ]:
SONDE_TAGS_SCAN = [("SONDE_Y025", 0.25), ("SONDE_Y050", 0.50), ("SONDE_Y075", 0.75)]

def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

scan = {"coarse": ("3D/coarse", case_3d.dataFileName_),
        "standard": ("3D/standard", cases_3d["standard"].dataFileName_),
        "fine": ("3D/fine", cases_3d["fine"].dataFileName_)}
scan_nodes = {"coarse": (9, 33, 9), "standard": (9, 65, 9), "fine": (9, 129, 9)}

print(f"level  nodes      max drift      max P0 repr err (bound)")
for lvl in ("coarse", "standard", "fine"):
    nx, ny, nz = scan_nodes[lvl]
    dy_half = 0.5 / (ny - 1)          # half-cell of this level
    worst_drift, worst_repr = 0.0, 0.0
    for tag, y_ref in SONDE_TAGS_SCAN:
        _, b = read_son_dir(scan[lvl][0], scan[lvl][1], tag)
        worst_drift = max(worst_drift, float(np.max(np.abs(b - b[0]))))
        worst_repr = max(worst_repr, abs(float(b[0]) - y_ref))
    print(f"{lvl:<9} {nx:>3}x{ny}x{nz:<3} {worst_drift:<13.2e} {worst_repr:.5f} ({dy_half + 1e-6:.5f})")
    assert worst_drift < 1e-8, f"equilibrium drift at {lvl}: {worst_drift:.3e}"
    assert worst_repr <= dy_half + 1e-6, f"P0 projection off at {lvl}: {worst_repr:.3e}"
print("OK: steady identity and the tightening P0 bound hold at every level.")

## Computer Performance


In [ ]:
run.tablePerf()